# ⚖️ AurumAI: The Gold Trading Council

A multi-agent AI paper trading system for the gold market, using **XAU/USD spot gold** as the primary instrument and gold futures/ETF proxies as confirmation.

### Architecture
```
MCP Data Servers          Specialist Agents           Arbitrator
─────────────────         ──────────────────          ──────────
gold_server.py    ──▶  📈 Technical Analyst   ──▶
news_server.py    ──▶  📰 Sentiment Analyst   ──▶   ⚖️  Arbitrator
macro_server.py   ──▶  🌍 Macro Economist     ──▶   (reads journal,
                        🛡️  Risk Manager        ──▶    tallies votes,
                        😈 Devil's Advocate    ──▶    sizes position)
```

### What makes this different
- Agents use **ReAct-style action/observation traces** without exposing private chain-of-thought
- A **Devil's Advocate** always argues the opposite of the majority
- A **Trading Journal** persists reasoning across the session
- **Confidence-weighted position sizing** — split council = smaller trade
- Live **HTML dashboard** served via Flask + ngrok

## 📦 Cell 1 — Install everything

In [ ]:
# ── CELL 1: Install all dependencies ──────────────────────────────────────────
!pip install openai fastmcp mcp feedparser requests flask pyngrok yfinance pandas numpy -q

print("✅ All packages installed")
print("  openai     — LLM calls")
print("  fastmcp    — MCP server framework")
print("  mcp        — MCP client")
print("  feedparser — Google News RSS")
print("  requests   — HTTP/FRED data calls")
print("  flask      — web server for dashboard")
print("  pyngrok    — public URL from Colab")
print("  yfinance   — real gold price data")
print("  pandas     — FRED CSV + data wrangling")
print("  numpy      — technical indicator math")
print("  sqlite3    — persistent memory, built into Python")


## 🔑 Cell 2 — API Keys

In [ ]:
# ── CELL 2: API Keys ───────────────────────────────────────────────────────────
# Option A: Paste directly (fine for a hackathon)
# Option B: Use Colab Secrets (sidebar 🔑 icon) — more secure

import os

# ▼ PASTE YOUR OPENAI KEY HERE
os.environ["OPENAI_API_KEY"] = "sk-YOUR-KEY-HERE"

# ▼ GET A FREE NGROK AUTH TOKEN AT https://dashboard.ngrok.com/get-started/your-authtoken
os.environ["NGROK_AUTH_TOKEN"] = "YOUR-NGROK-TOKEN-HERE"

# Verify
assert os.environ["OPENAI_API_KEY"] != "sk-YOUR-KEY-HERE", "❌ Please set your OpenAI key!"
assert os.environ["NGROK_AUTH_TOKEN"] != "YOUR-NGROK-TOKEN-HERE", "❌ Please set your ngrok token!"
print("✅ API keys set")

## 🧠 Cell 2B — Persistent Memory

SQLite-backed memory so the council remembers prior trades, lessons, macro regimes, and agent reflections across notebook sessions.


In [ ]:
# ── CELL 2B: Persistent Memory ────────────────────────────────────────────────
# Default backend: SQLite. It works in Colab without extra services.
# To persist across Colab runtimes, set AURUMAI_MEMORY_DB to a Google Drive path
# after mounting Drive, for example:
# os.environ["AURUMAI_MEMORY_DB"] = "/content/drive/MyDrive/aurumai_memory.sqlite3"

import os
import json
import sqlite3
from datetime import datetime
from pathlib import Path


class Memory:
    """Persistent memory for the AurumAI council.

    Tables map directly to the architecture:
    short_term_memory, long_term_memory, trade_history, lessons_learned,
    macro_history, and agent_reflections.
    """

    def __init__(self, db_path: str | None = None):
        self.db_path = db_path or os.environ.get("AURUMAI_MEMORY_DB", "aurumai_memory.sqlite3")
        Path(self.db_path).parent.mkdir(parents=True, exist_ok=True) if Path(self.db_path).parent != Path(".") else None
        self.conn = sqlite3.connect(self.db_path, check_same_thread=False)
        self.conn.row_factory = sqlite3.Row
        self._init_schema()

    def _init_schema(self):
        cur = self.conn.cursor()
        cur.execute("""CREATE TABLE IF NOT EXISTS short_term_memory (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            ts TEXT NOT NULL,
            agent TEXT,
            note TEXT NOT NULL
        )""")
        cur.execute("""CREATE TABLE IF NOT EXISTS long_term_memory (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            ts TEXT NOT NULL,
            key TEXT NOT NULL,
            value TEXT NOT NULL
        )""")
        cur.execute("""CREATE TABLE IF NOT EXISTS trade_history (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            ts TEXT NOT NULL,
            action TEXT,
            verdict TEXT,
            position_size_pct REAL,
            confidence_level TEXT,
            rationale TEXT,
            realized_outcome TEXT,
            pnl_pct REAL
        )""")
        cur.execute("""CREATE TABLE IF NOT EXISTS lessons_learned (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            ts TEXT NOT NULL,
            lesson TEXT NOT NULL,
            source TEXT
        )""")
        cur.execute("""CREATE TABLE IF NOT EXISTS macro_history (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            ts TEXT NOT NULL,
            snapshot_json TEXT NOT NULL
        )""")
        cur.execute("""CREATE TABLE IF NOT EXISTS agent_reflections (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            ts TEXT NOT NULL,
            agent TEXT NOT NULL,
            reflection TEXT NOT NULL
        )""")
        self.conn.commit()

    def _now(self) -> str:
        return datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC")

    def remember_short(self, agent: str, note: str):
        self.conn.execute(
            "INSERT INTO short_term_memory (ts, agent, note) VALUES (?, ?, ?)",
            (self._now(), agent, note),
        )
        self.conn.commit()

    def remember_long(self, key: str, value):
        payload = value if isinstance(value, str) else json.dumps(value, ensure_ascii=False)
        self.conn.execute(
            "INSERT INTO long_term_memory (ts, key, value) VALUES (?, ?, ?)",
            (self._now(), key, payload),
        )
        self.conn.commit()

    def add_trade(self, result: dict):
        self.conn.execute(
            """INSERT INTO trade_history
               (ts, action, verdict, position_size_pct, confidence_level, rationale)
               VALUES (?, ?, ?, ?, ?, ?)""",
            (
                self._now(),
                result.get("action"),
                result.get("final_verdict"),
                float(result.get("position_size_pct", 0) or 0),
                result.get("confidence_level"),
                result.get("entry_rationale"),
            ),
        )
        self.conn.commit()

    def add_lesson(self, lesson: str, source: str = "manual"):
        self.conn.execute(
            "INSERT INTO lessons_learned (ts, lesson, source) VALUES (?, ?, ?)",
            (self._now(), lesson, source),
        )
        self.conn.commit()

    def add_macro_snapshot(self, snapshot):
        payload = snapshot if isinstance(snapshot, str) else json.dumps(snapshot, ensure_ascii=False)
        self.conn.execute(
            "INSERT INTO macro_history (ts, snapshot_json) VALUES (?, ?)",
            (self._now(), payload),
        )
        self.conn.commit()

    def add_reflection(self, agent: str, reflection: str):
        self.conn.execute(
            "INSERT INTO agent_reflections (ts, agent, reflection) VALUES (?, ?, ?)",
            (self._now(), agent, reflection),
        )
        self.conn.commit()

    def recent(self, table: str, limit: int = 5) -> list[dict]:
        allowed = {
            "short_term_memory", "long_term_memory", "trade_history",
            "lessons_learned", "macro_history", "agent_reflections"
        }
        if table not in allowed:
            raise ValueError(f"Unsupported memory table: {table}")
        rows = self.conn.execute(
            f"SELECT * FROM {table} ORDER BY id DESC LIMIT ?",
            (limit,),
        ).fetchall()
        return [dict(row) for row in rows]

    def summary(self) -> str:
        parts = {
            "recent_trades": self.recent("trade_history", 5),
            "lessons_learned": self.recent("lessons_learned", 5),
            "macro_history": self.recent("macro_history", 3),
            "agent_reflections": self.recent("agent_reflections", 8),
        }
        return json.dumps(parts, indent=2, ensure_ascii=False)


memory = Memory()
print(f"✅ Persistent memory ready: {memory.db_path}")
print("Tables: short_term_memory, long_term_memory, trade_history, lessons_learned, macro_history, agent_reflections")


## 🛢️ Cell 3 — MCP Server: Gold Data
Fetches XAU/USD spot gold plus GC=F, MGC=F, and GLD confirmation data, then computes RSI, moving averages, trend, and volatility.

In [ ]:
%%writefile gold_server.py

# ============================================================
# GOLD DATA MCP SERVER
# Multi-instrument gold market view:
# - XAUUSD=X as the primary spot gold trading instrument
# - GC=F and MGC=F as futures confirmation
# - GLD as ETF confirmation
# ============================================================

from mcp.server.fastmcp import FastMCP
import yfinance as yf
import pandas as pd
import numpy as np
import json
from datetime import datetime

mcp = FastMCP("gold-data-server")

GOLD_INSTRUMENTS = {
    "XAUUSD=X": "Spot Gold XAU/USD",
    "GC=F": "COMEX Gold Futures",
    "MGC=F": "Micro Gold Futures",
    "GLD": "SPDR Gold ETF",
}


def compute_rsi(prices: pd.Series, period: int = 14) -> float:
    """Compute RSI for the last `period` candles."""
    if len(prices) <= period:
        return 50.0

    delta = prices.diff().dropna()
    gain = delta.where(delta > 0, 0.0)
    loss = -delta.where(delta < 0, 0.0)
    avg_gain = gain.rolling(window=period).mean().iloc[-1]
    avg_loss = loss.rolling(window=period).mean().iloc[-1]

    if pd.isna(avg_gain) or pd.isna(avg_loss):
        return 50.0
    if avg_loss == 0:
        return 100.0

    rs = avg_gain / avg_loss
    return round(100 - (100 / (1 + rs)), 2)


def instrument_snapshot(symbol: str, name: str) -> dict:
    """Fetch price, trend, RSI, moving averages, and volatility for one instrument."""
    ticker = yf.Ticker(symbol)
    hist = ticker.history(period="3mo", interval="1d")

    if hist.empty or "Close" not in hist:
        return {
            "symbol": symbol,
            "name": name,
            "error": "No data returned",
        }

    closes = hist["Close"].dropna()
    if len(closes) < 2:
        return {
            "symbol": symbol,
            "name": name,
            "error": "Not enough price history",
        }

    current_price = round(float(closes.iloc[-1]), 4)
    prev_price = round(float(closes.iloc[-2]), 4)
    change_pct = round(((current_price - prev_price) / prev_price) * 100, 3)

    sma20 = round(float(closes.rolling(20).mean().iloc[-1]), 4) if len(closes) >= 20 else current_price
    sma50 = round(float(closes.rolling(50).mean().iloc[-1]), 4) if len(closes) >= 50 else current_price
    rsi = compute_rsi(closes)

    recent = closes.tail(10).round(4).tolist()
    trend = "UP" if recent[-1] > recent[0] else "DOWN"

    log_returns = np.log(closes / closes.shift(1)).dropna()
    vol_30d = round(float(log_returns.tail(30).std() * np.sqrt(252) * 100), 2) if len(log_returns) else 0.0

    return {
        "symbol": symbol,
        "name": name,
        "current_price": current_price,
        "prev_close": prev_price,
        "change_24h_pct": change_pct,
        "rsi_14": rsi,
        "rsi_signal": "OVERBOUGHT" if rsi > 70 else ("OVERSOLD" if rsi < 30 else "NEUTRAL"),
        "sma_20": sma20,
        "sma_50": sma50,
        "price_above_sma20": current_price > sma20,
        "price_above_sma50": current_price > sma50,
        "golden_cross_active": sma20 > sma50,
        "10d_price_history": recent,
        "10d_trend": trend,
        "annualised_volatility_pct": vol_30d,
        "vol_regime": "HIGH" if vol_30d > 20 else ("LOW" if vol_30d < 10 else "NORMAL"),
    }


@mcp.tool()
def get_gold_market_snapshot() -> str:
    """
    Multi-instrument gold market snapshot.
    Primary trading instrument: XAU/USD spot gold.
    Confirmation instruments: GC=F, MGC=F, GLD.
    """
    try:
        instruments = {}
        for symbol, name in GOLD_INSTRUMENTS.items():
            try:
                instruments[symbol] = instrument_snapshot(symbol, name)
            except Exception as e:
                instruments[symbol] = {
                    "symbol": symbol,
                    "name": name,
                    "error": str(e),
                }

        result = {
            "market": "Gold Market",
            "primary_instrument": "XAUUSD=X",
            "primary_description": "Spot Gold XAU/USD, not physical jewelry gold",
            "confirmation_instruments": ["GC=F", "MGC=F", "GLD"],
            "timestamp": datetime.utcnow().strftime("%Y-%m-%d %H:%M UTC"),
            "instruments": instruments,
        }
        return json.dumps(result, indent=2)
    except Exception as e:
        return f"ERROR fetching gold market snapshot: {e}"


@mcp.tool()
def get_gold_technicals() -> str:
    """
    Backward-compatible tool.
    Returns the main XAU/USD spot gold technicals.
    """
    try:
        result = instrument_snapshot("XAUUSD=X", "Spot Gold XAU/USD")
        result["asset"] = "Spot Gold XAU/USD"
        result["timestamp"] = datetime.utcnow().strftime("%Y-%m-%d %H:%M UTC")
        result["current_price_usd"] = result.get("current_price")
        result["prev_close_usd"] = result.get("prev_close")
        return json.dumps(result, indent=2)
    except Exception as e:
        return f"ERROR fetching gold technicals: {e}"


@mcp.tool()
def get_gold_volatility() -> str:
    """
    Backward-compatible tool.
    Returns volatility for XAU/USD spot gold.
    """
    try:
        result = instrument_snapshot("XAUUSD=X", "Spot Gold XAU/USD")
        return json.dumps({
            "instrument": "XAUUSD=X",
            "name": "Spot Gold XAU/USD",
            "annualised_volatility_pct": result.get("annualised_volatility_pct"),
            "vol_regime": result.get("vol_regime"),
        }, indent=2)
    except Exception as e:
        return f"ERROR computing volatility: {e}"


if __name__ == "__main__":
    mcp.run()


## 📰 Cell 4 — MCP Server: News & Sentiment

In [ ]:
%%writefile news_server.py

# ============================================================
# NEWS & SENTIMENT MCP SERVER
# Google News RSS, including source-targeted queries for
# Reuters, Bloomberg, and Al Jazeera when those feeds are visible.
# ============================================================

from mcp.server.fastmcp import FastMCP
import feedparser
import json
from datetime import datetime

mcp = FastMCP("news-sentiment-server")

QUERIES = {
    "gold_price": "gold+price+trading",
    "gold_macro": "gold+inflation+federal+reserve",
    "usd_strength": "US+dollar+index+DXY+strength",
    "geopolitical": "geopolitical+risk+safe+haven+gold",
    "reuters_gold": "gold+markets+site:reuters.com",
    "bloomberg_gold": "gold+markets+site:bloomberg.com",
    "aljazeera_gold": "gold+geopolitics+site:aljazeera.com",
}


def fetch_headlines(query: str, limit: int = 5) -> list:
    url = f"https://news.google.com/rss/search?q={query}&hl=en-US&gl=US&ceid=US:en"
    feed = feedparser.parse(url)
    headlines = []
    for entry in feed.entries[:limit]:
        headlines.append({
            "title": entry.title,
            "published": entry.get("published", "unknown"),
            "source": entry.get("source", {}).get("title", "Google News"),
            "link": entry.get("link", ""),
        })
    return headlines


@mcp.tool()
def get_gold_news() -> str:
    '''Fetch latest gold price and gold-macro headlines from Google News RSS.'''
    try:
        results = {
            "timestamp": datetime.utcnow().strftime("%Y-%m-%d %H:%M UTC"),
            "gold_price_news": fetch_headlines(QUERIES["gold_price"]),
            "gold_macro_news": fetch_headlines(QUERIES["gold_macro"]),
        }
        return json.dumps(results, indent=2)
    except Exception as e:
        return f"ERROR fetching gold news: {e}"


@mcp.tool()
def get_source_news() -> str:
    '''Fetch source-targeted gold headlines for Reuters, Bloomberg, and Al Jazeera.'''
    try:
        results = {
            "timestamp": datetime.utcnow().strftime("%Y-%m-%d %H:%M UTC"),
            "reuters": fetch_headlines(QUERIES["reuters_gold"], limit=4),
            "bloomberg": fetch_headlines(QUERIES["bloomberg_gold"], limit=4),
            "al_jazeera": fetch_headlines(QUERIES["aljazeera_gold"], limit=4),
        }
        return json.dumps(results, indent=2)
    except Exception as e:
        return f"ERROR fetching source news: {e}"


@mcp.tool()
def get_macro_news() -> str:
    '''
    Fetch USD strength and geopolitical risk headlines.
    Gold is inversely correlated with USD and a safe-haven in geopolitical risk.
    '''
    try:
        results = {
            "timestamp": datetime.utcnow().strftime("%Y-%m-%d %H:%M UTC"),
            "usd_headlines": fetch_headlines(QUERIES["usd_strength"]),
            "geopolitical_headlines": fetch_headlines(QUERIES["geopolitical"]),
        }
        return json.dumps(results, indent=2)
    except Exception as e:
        return f"ERROR fetching macro news: {e}"


if __name__ == "__main__":
    mcp.run()


## 🌍 Cell 5 — MCP Server: Macro Indicators

In [ ]:
%%writefile macro_server.py

# ============================================================
# MACRO INDICATORS MCP SERVER
# Yahoo Finance market indicators + FRED inflation series.
# FRED CSV endpoints do not require an API key.
# ============================================================

from mcp.server.fastmcp import FastMCP
import yfinance as yf
import pandas as pd
import requests
import json
from datetime import datetime
from io import StringIO

mcp = FastMCP("macro-indicators-server")

FRED_SERIES = {
    "CPIAUCSL": "US CPI",
    "CPILFESL": "Core CPI",
    "PCEPI": "PCE Price Index",
    "PCEPILFE": "Core PCE Price Index",
}


def fetch_ticker_summary(symbol: str, name: str) -> dict:
    try:
        t = yf.Ticker(symbol)
        hist = t.history(period="5d", interval="1d")
        if hist.empty:
            return {"symbol": symbol, "error": "no data"}
        closes = hist["Close"]
        current = round(float(closes.iloc[-1]), 4)
        prev = round(float(closes.iloc[-2]), 4)
        change_pct = round(((current - prev) / prev) * 100, 3)
        return {
            "symbol": symbol,
            "name": name,
            "current": current,
            "change_24h_pct": change_pct,
            "direction": "UP" if change_pct > 0 else "DOWN",
        }
    except Exception as e:
        return {"symbol": symbol, "error": str(e)}


def fetch_fred_series(series_id: str, label: str) -> dict:
    try:
        url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
        response = requests.get(url, timeout=20)
        response.raise_for_status()
        df = pd.read_csv(StringIO(response.text))
        df = df.rename(columns={"observation_date": "date", series_id: "value"})
        df["value"] = pd.to_numeric(df["value"], errors="coerce")
        df = df.dropna().tail(14)
        latest = df.iloc[-1]
        prev = df.iloc[-2]
        year_ago = df.iloc[-13] if len(df) >= 13 else None
        mom_pct = round(((latest["value"] - prev["value"]) / prev["value"]) * 100, 3)
        yoy_pct = None
        if year_ago is not None:
            yoy_pct = round(((latest["value"] - year_ago["value"]) / year_ago["value"]) * 100, 3)
        return {
            "series_id": series_id,
            "name": label,
            "latest_date": str(latest["date"]),
            "latest_value": round(float(latest["value"]), 3),
            "month_over_month_pct": mom_pct,
            "year_over_year_pct": yoy_pct,
            "source": "FRED",
        }
    except Exception as e:
        return {"series_id": series_id, "name": label, "error": str(e), "source": "FRED"}


@mcp.tool()
def get_macro_indicators() -> str:
    '''
    Fetch key macro indicators that drive gold price:
    - DX-Y.NYB: US Dollar Index (gold moves OPPOSITE to USD)
    - TLT: 20-Year Treasury Bond ETF (bonds up = risk-off = gold up)
    - ^VIX: Volatility Index (fear gauge — high VIX = gold demand)
    - ^TNX: 10-Year Treasury Yield (high yield = lower gold)
    '''
    try:
        indicators = [
            fetch_ticker_summary("DX-Y.NYB", "US Dollar Index (DXY)"),
            fetch_ticker_summary("TLT", "20Y Treasury Bond ETF"),
            fetch_ticker_summary("^VIX", "CBOE Volatility Index (VIX)"),
            fetch_ticker_summary("^TNX", "10Y Treasury Yield"),
        ]
        result = {
            "timestamp": datetime.utcnow().strftime("%Y-%m-%d %H:%M UTC"),
            "macro_indicators": indicators,
            "gold_macro_rules": {
                "DXY": "Gold moves OPPOSITE to USD — DXY up is bearish for gold",
                "TLT": "TLT up = risk-off sentiment = bullish for gold",
                "VIX": "High VIX = market fear = safe-haven demand for gold",
                "TNX": "High real yields = bearish for gold (opportunity cost)",
            },
        }
        return json.dumps(result, indent=2)
    except Exception as e:
        return f"ERROR fetching macro indicators: {e}"


@mcp.tool()
def get_inflation_snapshot() -> str:
    '''Fetch CPI, Core CPI, PCE, and Core PCE from FRED.'''
    try:
        result = {
            "timestamp": datetime.utcnow().strftime("%Y-%m-%d %H:%M UTC"),
            "source": "FRED public CSV",
            "inflation_series": [
                fetch_fred_series(series_id, label)
                for series_id, label in FRED_SERIES.items()
            ],
            "gold_macro_rules": {
                "inflation_up": "Rising inflation can support gold if real yields do not rise faster",
                "inflation_down": "Cooling inflation can reduce safe-haven/inflation-hedge demand",
                "fed_reaction": "Gold reacts most to real rates and expected Fed policy, not CPI alone",
            },
        }
        return json.dumps(result, indent=2)
    except Exception as e:
        return f"ERROR fetching inflation snapshot: {e}"


if __name__ == "__main__":
    mcp.run()


## 🔌 Cell 6 — MCP Client: call any tool on any server

In [ ]:
# ── CELL 6: MCP Client utility ─────────────────────────────────────────────────
# Reusable async function that spins up any server, calls a tool, returns result.

import asyncio
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


async def call_tool(server_script: str, tool_name: str, arguments: dict = {}) -> str:
    '''
    Universal MCP tool caller.
    Spins up server_script as a subprocess, calls tool_name with arguments,
    returns the result as a string.
    '''
    server_params = StdioServerParameters(
        command=sys.executable,
        args=[server_script],
    )
    with open("mcp_errors.log", "a") as log:
        async with stdio_client(server_params, errlog=log) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                result = await session.call_tool(tool_name, arguments)
                parts = []
                for item in result.content:
                    parts.append(item.text if hasattr(item, "text") else str(item))
                return "\n".join(parts)


async def collect_all_data() -> dict:
    '''Fire all MCP tool calls and return a dict of raw results.'''
    print("🔌 Calling MCP servers...")

    calls = {
        "gold_market_snapshot": call_tool("gold_server.py",  "get_gold_market_snapshot"),
        "gold_technicals":    call_tool("gold_server.py",  "get_gold_technicals"),
        "gold_volatility":    call_tool("gold_server.py",  "get_gold_volatility"),
        "gold_news":          call_tool("news_server.py",  "get_gold_news"),
        "source_news":        call_tool("news_server.py",  "get_source_news"),
        "macro_news":         call_tool("news_server.py",  "get_macro_news"),
        "macro_indicators":   call_tool("macro_server.py", "get_macro_indicators"),
        "inflation_snapshot": call_tool("macro_server.py", "get_inflation_snapshot"),
    }

    results = {}
    for name, coro in calls.items():
        try:
            results[name] = await coro
            print(f"  ✅ {name}")
        except Exception as e:
            results[name] = f"FAILED: {e}"
            print(f"  ❌ {name}: {e}")

    try:
        memory.add_macro_snapshot({
            "macro_indicators": results.get("macro_indicators"),
            "inflation_snapshot": results.get("inflation_snapshot"),
        })
    except Exception as e:
        print(f"  ⚠️ Memory macro snapshot skipped: {e}")

    return results


# Quick test
test_data = await collect_all_data()
print("\n📊 Gold Market Snapshot preview:")
print(test_data["gold_market_snapshot"][:500], "...")


## 🤖 Cell 7 — The Five Specialist Agents

In [ ]:
# ── CELL 7: Agent definitions ──────────────────────────────────────────────────
import json
from openai import OpenAI
from datetime import datetime

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
MODEL = "gpt-4o-mini"

# ── Trading Journal: short-term session view backed by SQLite memory ───────────
trading_journal: list[str] = []


def journal_entry(agent_name: str, note: str):
    ts = datetime.utcnow().strftime("%H:%M:%S")
    entry = f"[{ts}] {agent_name}: {note}"
    trading_journal.append(entry)
    try:
        memory.remember_short(agent_name, note)
    except Exception as e:
        print(f"     ⚠️ Memory write skipped: {e}")


def memory_context() -> str:
    try:
        return memory.summary()
    except Exception as e:
        return f"Memory unavailable: {e}"


# ── Verdict structure ─────────────────────────────────────────────────────────
VERDICT_SCHEMA = '''
You MUST reply with ONLY valid JSON in this exact shape, no other text:
{
  "verdict": "BULLISH" | "BEARISH" | "NEUTRAL",
  "confidence": <integer 1-10>,
  "summary": "<one sentence: your key reason>",
  "key_signal": "<the single most important data point you saw>",
  "react_trace": [
    {"action": "<tool/data source consulted>", "observation": "<brief observation>"}
  ],
  "journal_note": "<10 words max for the trading journal>",
  "reflection": "<one short lesson this agent should remember>"
}
'''


def run_agent(name: str, system_prompt: str, data: str, planned_actions: list[str]) -> dict:
    '''
    ReAct-style agent call:
    1. The prompt gives the agent a plan of actions/data sources to consult.
    2. The agent reads observations from MCP tool outputs.
    3. The agent returns a final structured vote plus a compact action/observation trace.
    '''
    print(f"  🤖 {name} is deliberating with ReAct-style tool observations...")
    action_plan = "\n".join([f"- Action: {a}" for a in planned_actions])
    prompt = f'''Use this ReAct-style decision protocol:
1. Decide which listed action/data source matters most.
2. Read the observation data.
3. If the data conflicts, weigh the conflict explicitly.
4. Return the final JSON verdict only.

Do not reveal hidden chain-of-thought. The react_trace field should contain concise
action/observation notes, not private reasoning.

AVAILABLE ACTIONS:
{action_plan}

PERSISTENT MEMORY CONTEXT:
{memory_context()}

CURRENT OBSERVATIONS:
{data}
'''
    response = client.chat.completions.create(
        model=MODEL,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": system_prompt + "\n\n" + VERDICT_SCHEMA},
            {"role": "user", "content": prompt},
        ],
    )
    raw = response.choices[0].message.content
    verdict = json.loads(raw)
    verdict["agent"] = name
    journal_entry(name, verdict.get("journal_note", verdict["summary"][:60]))
    if verdict.get("reflection"):
        memory.add_reflection(name, verdict["reflection"])
    print(f"     → {verdict['verdict']} (confidence: {verdict['confidence']}/10)")
    return verdict


# ── Agent 1: Technical Analyst ────────────────────────────────────────────────
def technical_analyst(data: dict) -> dict:
    system = '''You are a quantitative technical analyst specialising in gold futures (GC=F).
Your ONLY job: analyse price action, RSI, and moving averages from the data provided.
Your primary trading instrument is XAU/USD spot gold. Use GC=F, MGC=F, and GLD as confirmation markets.
Focus on: RSI overbought/oversold, golden/death cross, price vs SMAs, trend direction.
Use memory to notice repeated technical patterns. Ignore news entirely.'''
    return run_agent(
        "📈 Technical Analyst",
        system,
        f"GOLD MARKET SNAPSHOT:\n{data['gold_market_snapshot']}\n\nPRIMARY XAU/USD TECHNICALS:\n{data['gold_technicals']}\n\nVOLATILITY:\n{data['gold_volatility']}",
        ["get_gold_market_snapshot", "get_gold_technicals", "get_gold_volatility", "read technical pattern memory"],
    )


# ── Agent 2: Sentiment Analyst ────────────────────────────────────────────────
def sentiment_analyst(data: dict) -> dict:
    system = '''You are a financial news sentiment analyst specialising in gold markets.
Your ONLY job: read the news headlines provided and gauge the market mood.
Focus on tone, source mix, recurring themes, and whether news flow is net positive
or negative for gold demand. Ignore charts.'''
    return run_agent(
        "📰 Sentiment Analyst",
        system,
        f"GOLD NEWS HEADLINES:\n{data['gold_news']}\n\nSOURCE-TARGETED NEWS:\n{data['source_news']}\n\nMACRO NEWS:\n{data['macro_news']}",
        ["get_gold_news", "get_source_news", "get_macro_news", "read sentiment memory"],
    )


# ── Agent 3: Macro Economist ──────────────────────────────────────────────────
def macro_economist(data: dict) -> dict:
    system = '''You are a macro economist specialising in gold's relationship with monetary policy.
Your ONLY job: analyse USD strength, bonds, yields, volatility, CPI, Core CPI, PCE,
Core PCE, and likely Fed reaction. Apply these rules:
  - DXY rising = bearish for gold
  - Real yields rising = bearish for gold
  - VIX high = bullish for gold
  - TLT rising = risk-off = bullish for gold
  - Inflation supports gold only when real-rate expectations do not overwhelm it.'''
    return run_agent(
        "🌍 Macro Economist",
        system,
        f"MACRO INDICATORS:\n{data['macro_indicators']}\n\nINFLATION SNAPSHOT:\n{data['inflation_snapshot']}\n\nMACRO NEWS CONTEXT:\n{data['macro_news']}",
        ["get_macro_indicators", "get_inflation_snapshot", "get_macro_news", "read macro regime memory"],
    )


# ── Agent 4: Risk Manager ─────────────────────────────────────────────────────
def risk_manager(data: dict) -> dict:
    system = '''You are a risk manager at a hedge fund responsible for gold position sizing.
Your ONLY job: assess whether current conditions make it SAFE to hold a large gold position.
Focus on volatility regime, extreme RSI, macro shocks, and historical trade outcomes.
You are conservative by nature. Your verdict is about risk level, not price direction.'''
    return run_agent(
        "🛡️ Risk Manager",
        system,
        f"GOLD MARKET SNAPSHOT:\n{data['gold_market_snapshot']}\n\nVOLATILITY DATA:\n{data['gold_volatility']}\n\nTECHNICAL DATA:\n{data['gold_technicals']}\n\nMACRO:\n{data['macro_indicators']}\n\nINFLATION:\n{data['inflation_snapshot']}",
        ["get_gold_volatility", "get_gold_technicals", "get_macro_indicators", "read trade history"],
    )


# ── Agent 5: Devil's Advocate ─────────────────────────────────────────────────
def devils_advocate(verdicts: list[dict]) -> dict:
    '''
    The Devil's Advocate reads the other agents' verdicts
    and argues the OPPOSITE of the majority view.
    '''
    votes = [v["verdict"] for v in verdicts]
    majority = max(set(votes), key=votes.count)
    opposite = "BEARISH" if majority == "BULLISH" else ("BULLISH" if majority == "BEARISH" else "BEARISH")

    verdicts_text = "\n".join([
        f"- {v['agent']}: {v['verdict']} (confidence {v['confidence']}/10) — {v['summary']}"
        for v in verdicts
    ])

    system = f'''You are the Devil's Advocate on a trading council.
The other four agents have voted. Your ONLY job is to argue AGAINST the majority view.
The majority voted: {majority}
You MUST argue: {opposite}
Find the strongest possible counter-argument to the consensus.
Be provocative but grounded in real financial reasoning.'''

    return run_agent(
        "😈 Devil's Advocate",
        system,
        f"OTHER AGENTS' VERDICTS:\n{verdicts_text}",
        ["read specialist verdicts", "read prior failure modes from memory", "construct counter-thesis"],
    )


print("✅ All 5 agents defined with ReAct-style action/observation traces")


## ⚖️ Cell 8 — The Arbitrator

In [ ]:
# ── CELL 8: Arbitrator ─────────────────────────────────────────────────────────
# Reads all verdicts + the trading journal.
# Tallies votes, weighs confidence, sizes position.
# Produces the final actionable recommendation.

def tally_votes(verdicts: list[dict]) -> dict:
    """Count votes and compute weighted confidence score."""
    tally = {"BULLISH": 0, "BEARISH": 0, "NEUTRAL": 0}
    weighted = {"BULLISH": 0, "BEARISH": 0, "NEUTRAL": 0}

    for v in verdicts:
        verdict = v["verdict"]
        confidence = v["confidence"]
        tally[verdict] += 1
        weighted[verdict] += confidence

    return {"vote_count": tally, "weighted_confidence": weighted}


def compute_position_size(tally: dict, all_verdicts: list[dict]) -> dict:
    """
    Confidence-weighted position sizing.
    The more unanimous and confident the council, the larger the suggested position.
    Split council = small or no trade.
    """
    votes = tally["vote_count"]
    weighted = tally["weighted_confidence"]
    total_agents = len(all_verdicts)  # including devil's advocate

    bull = votes["BULLISH"]
    bear = votes["BEARISH"]
    neutral = votes["NEUTRAL"]

    # Unanimity score: 0 (completely split) to 1 (fully unanimous)
    majority_count = max(bull, bear, neutral)
    unanimity = majority_count / total_agents

    # Average confidence of majority voters
    majority_verdict = max(votes, key=votes.get)
    majority_votes = [v for v in all_verdicts if v["verdict"] == majority_verdict]
    avg_confidence = sum(v["confidence"] for v in majority_votes) / max(len(majority_votes), 1)

    # Position size: max 5% of portfolio, scaled by unanimity × confidence
    max_position_pct = 5.0
    position_pct = round(max_position_pct * unanimity * (avg_confidence / 10), 2)

    if position_pct < 0.5 or majority_verdict == "NEUTRAL":
        action = "HOLD / PAPER TRADE ONLY"
        position_pct = 0.0
    elif majority_verdict == "BULLISH":
        action = "BUY"
    else:
        action = "SELL / SHORT"

    return {
        "action": action,
        "position_size_pct": position_pct,
        "majority_verdict": majority_verdict,
        "unanimity_score": round(unanimity, 2),
        "avg_majority_confidence": round(avg_confidence, 1),
        "council_split": f"{bull}B/{bear}Be/{neutral}N",
    }


def arbitrator(all_verdicts: list[dict], data: dict) -> dict:
    """
    The Arbitrator reads all verdicts + the trading journal
    and produces the final structured recommendation.
    """
    print("\n  ⚖️  Arbitrator convening...")

    tally = tally_votes(all_verdicts)
    position = compute_position_size(tally, all_verdicts)

    verdicts_text = "\n".join([
        f"- {v['agent']}: {v['verdict']} ({v['confidence']}/10) — {v['summary']}"
        for v in all_verdicts
    ])
    journal_text = "\n".join(trading_journal)

    devil = next((v for v in all_verdicts if "Devil" in v["agent"]), None)
    devil_text = f"\nDevil's Advocate argued: {devil['summary']}" if devil else ""

    prompt = f"""You are the Arbitrator of the Gold Trading Council.

COUNCIL VERDICTS:
{verdicts_text}

VOTE TALLY: {tally['vote_count']}
WEIGHTED CONFIDENCE: {tally['weighted_confidence']}
POSITION SIZING SYSTEM OUTPUT: {position}
{devil_text}

TRADING JOURNAL (current session short-term memory):
{journal_text}

LONG-TERM MEMORY CONTEXT:
{memory_context()}

Write a final recommendation in this EXACT JSON format:
{{
  "final_verdict": "BULLISH" | "BEARISH" | "NEUTRAL",
  "action": "BUY" | "SELL" | "HOLD",
  "position_size_pct": <float>,
  "confidence_level": "HIGH" | "MEDIUM" | "LOW",
  "entry_rationale": "<2-3 sentences: why this trade makes sense now>",
  "devil_response": "<1 sentence: how you weighed the devil's counter-argument>",
  "stop_loss_note": "<1 sentence: where/when to cut the trade>",
  "council_summary": "<1 sentence: describe the council dynamic, e.g. 3-1 majority with dissent>"
}}"""

    response = client.chat.completions.create(
        model=MODEL,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": prompt}],
    )
    result = json.loads(response.choices[0].message.content)
    result["vote_tally"] = tally["vote_count"]
    result["position_calc"] = position
    result["all_verdicts"] = all_verdicts
    result["journal"] = trading_journal.copy()
    result["memory_db"] = memory.db_path
    memory.add_trade(result)
    memory.remember_long("last_council_result", result)

    journal_entry("⚖️ Arbitrator", f"Final: {result['action']} {result['position_size_pct']}% — {result['council_summary'][:50]}")
    print(f"     → Final: {result['action']} | {result['final_verdict']} | Position: {result['position_size_pct']}%")
    return result


print("✅ Arbitrator defined")

## 🏛️ Cell 9 — Run the Full Council Session

In [ ]:
# ── CELL 9: Run the full council session ───────────────────────────────────────

async def run_council_session() -> dict:
    """
    Full pipeline:
    1. Collect data from all MCP servers
    2. Run 4 specialist agents with ReAct-style action/observation traces
    3. Devil's Advocate argues against the majority
    4. Arbitrator tallies, weighs, and produces final recommendation
    """
    global trading_journal
    trading_journal = []   # reset visible session journal; SQLite memory persists

    print("═" * 55)
    print("⚖️  AURUMAI — THE GOLD TRADING COUNCIL")
    print("═" * 55)

    # ── Phase 1: Data collection ─────────────────────────────
    print("\n📡 PHASE 1: Collecting live market data...")
    data = await collect_all_data()

    # ── Phase 2: Specialist agents (run one by one — Colab event loop) ──────────
    # Note: We run sequentially here to avoid Colab asyncio conflicts.
    # Each agent gets only its relevant data — no cross-contamination.
    print("\n🤖 PHASE 2: Council deliberating...")
    v1 = technical_analyst(data)
    v2 = sentiment_analyst(data)
    v3 = macro_economist(data)
    v4 = risk_manager(data)
    specialist_verdicts = [v1, v2, v3, v4]

    # ── Phase 3: Devil's Advocate ─────────────────────────────
    print("\n😈 PHASE 3: Devil's Advocate preparing counter-argument...")
    v5 = devils_advocate(specialist_verdicts)
    all_verdicts = specialist_verdicts + [v5]

    # ── Phase 4: Arbitrator ───────────────────────────────────
    print("\n⚖️  PHASE 4: Arbitrator rendering final verdict...")
    final = arbitrator(all_verdicts, data)

    print("\n" + "═" * 55)
    print(f"  ACTION:    {final['action']}")
    print(f"  VERDICT:   {final['final_verdict']}")
    print(f"  POSITION:  {final['position_size_pct']}% of portfolio")
    print(f"  COUNCIL:   {final['council_summary']}")
    print("═" * 55)

    return final


# Run it!
session_result = await run_council_session()

## 🌐 Cell 10 — HTML Dashboard

In [ ]:
%%writefile dashboard.html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>AurumAI — The Gold Trading Council</title>
<style>
  /* ── Design tokens ── */
  :root {
    --gold:       #C9A84C;
    --gold-light: #E8CC7A;
    --gold-dim:   #7A6130;
    --bg:         #0A0A0B;
    --surface:    #111114;
    --surface2:   #18181D;
    --border:     #2A2A32;
    --text:       #E8E8EC;
    --text-dim:   #7A7A8A;
    --bullish:    #3DDC84;
    --bearish:    #FF4D6D;
    --neutral:    #8888AA;
    --devil:      #C850FF;
    --font-mono:  'JetBrains Mono', 'Fira Mono', monospace;
    --font-body:  'Inter', system-ui, sans-serif;
  }

  * { box-sizing: border-box; margin: 0; padding: 0; }

  body {
    background: var(--bg);
    color: var(--text);
    font-family: var(--font-body);
    min-height: 100vh;
    padding: 0 0 4rem;
  }

  /* ── Header ── */
  header {
    background: var(--surface);
    border-bottom: 1px solid var(--border);
    padding: 1.25rem 2rem;
    display: flex;
    align-items: center;
    justify-content: space-between;
    position: sticky;
    top: 0;
    z-index: 100;
  }
  .logo {
    font-family: var(--font-mono);
    font-size: 1.1rem;
    color: var(--gold);
    letter-spacing: 0.12em;
    text-transform: uppercase;
  }
  .logo span { color: var(--text-dim); font-size: 0.75rem; letter-spacing: 0.06em; }
  #status-badge {
    font-family: var(--font-mono);
    font-size: 0.72rem;
    padding: 0.3rem 0.85rem;
    border-radius: 2px;
    border: 1px solid var(--border);
    color: var(--text-dim);
    letter-spacing: 0.1em;
  }
  #status-badge.live { border-color: var(--bullish); color: var(--bullish); }
  #status-badge.running { border-color: var(--gold); color: var(--gold); }

  /* ── Main layout ── */
  main { max-width: 1200px; margin: 2rem auto; padding: 0 1.5rem; }

  /* ── Verdict Hero ── */
  #verdict-hero {
    background: var(--surface);
    border: 1px solid var(--border);
    border-top: 3px solid var(--gold);
    padding: 2rem 2.5rem;
    margin-bottom: 1.5rem;
    display: grid;
    grid-template-columns: 1fr 1fr 1fr 1fr;
    gap: 2rem;
    align-items: center;
  }
  .hero-action {
    grid-column: 1;
  }
  .action-label {
    font-family: var(--font-mono);
    font-size: 0.65rem;
    letter-spacing: 0.2em;
    color: var(--text-dim);
    text-transform: uppercase;
    margin-bottom: 0.4rem;
  }
  .action-value {
    font-family: var(--font-mono);
    font-size: 2.8rem;
    font-weight: 700;
    line-height: 1;
    letter-spacing: 0.04em;
  }
  .action-value.BULLISH, .action-value.BUY   { color: var(--bullish); }
  .action-value.BEARISH, .action-value.SELL  { color: var(--bearish); }
  .action-value.NEUTRAL, .action-value.HOLD  { color: var(--neutral); }

  .hero-stat { display: flex; flex-direction: column; gap: 0.3rem; }
  .stat-label {
    font-family: var(--font-mono);
    font-size: 0.62rem;
    letter-spacing: 0.18em;
    color: var(--text-dim);
    text-transform: uppercase;
  }
  .stat-value {
    font-family: var(--font-mono);
    font-size: 1.5rem;
    color: var(--text);
  }
  .stat-value.gold { color: var(--gold); }
  .rationale {
    grid-column: 2 / 5;
    font-size: 0.92rem;
    color: var(--text-dim);
    line-height: 1.6;
    border-left: 2px solid var(--border);
    padding-left: 1.5rem;
  }

  /* ── Run button ── */
  #run-btn {
    display: block;
    width: 100%;
    padding: 1rem;
    background: transparent;
    border: 1px solid var(--gold-dim);
    color: var(--gold);
    font-family: var(--font-mono);
    font-size: 0.85rem;
    letter-spacing: 0.18em;
    text-transform: uppercase;
    cursor: pointer;
    margin-bottom: 1.5rem;
    transition: all 0.2s;
  }
  #run-btn:hover { background: var(--gold-dim); color: var(--bg); }
  #run-btn:disabled { opacity: 0.4; cursor: not-allowed; }

  /* ── Grid layout ── */
  .grid-2 { display: grid; grid-template-columns: 1fr 1fr; gap: 1.5rem; margin-bottom: 1.5rem; }
  .grid-3 { display: grid; grid-template-columns: 1fr 1fr 1fr; gap: 1.5rem; margin-bottom: 1.5rem; }

  /* ── Cards ── */
  .card {
    background: var(--surface);
    border: 1px solid var(--border);
    padding: 1.4rem 1.6rem;
  }
  .card-title {
    font-family: var(--font-mono);
    font-size: 0.65rem;
    letter-spacing: 0.2em;
    text-transform: uppercase;
    color: var(--text-dim);
    margin-bottom: 1rem;
    padding-bottom: 0.6rem;
    border-bottom: 1px solid var(--border);
  }

  /* ── Agent verdict cards ── */
  .agent-card {
    background: var(--surface2);
    border: 1px solid var(--border);
    padding: 1.2rem 1.4rem;
    position: relative;
  }
  .agent-card.devil { border-color: var(--devil); }

  .agent-header {
    display: flex;
    justify-content: space-between;
    align-items: flex-start;
    margin-bottom: 0.7rem;
  }
  .agent-name {
    font-size: 0.82rem;
    font-weight: 600;
    color: var(--text);
  }
  .agent-verdict {
    font-family: var(--font-mono);
    font-size: 0.7rem;
    font-weight: 700;
    letter-spacing: 0.12em;
    padding: 0.2rem 0.6rem;
  }
  .agent-verdict.BULLISH { color: var(--bullish); border: 1px solid var(--bullish); }
  .agent-verdict.BEARISH { color: var(--bearish); border: 1px solid var(--bearish); }
  .agent-verdict.NEUTRAL { color: var(--neutral); border: 1px solid var(--neutral); }

  .agent-summary { font-size: 0.82rem; color: var(--text-dim); line-height: 1.5; margin-bottom: 0.7rem; }
  .agent-key-signal {
    font-family: var(--font-mono);
    font-size: 0.72rem;
    color: var(--gold-dim);
    margin-bottom: 0.5rem;
  }

  /* ── Confidence bar ── */
  .confidence-bar-wrap {
    display: flex;
    align-items: center;
    gap: 0.6rem;
  }
  .confidence-bar-bg {
    flex: 1;
    height: 3px;
    background: var(--border);
  }
  .confidence-bar-fill {
    height: 100%;
    background: var(--gold);
    transition: width 0.8s ease;
  }
  .confidence-num {
    font-family: var(--font-mono);
    font-size: 0.68rem;
    color: var(--text-dim);
    white-space: nowrap;
  }

  /* ── Vote tally bar ── */
  .vote-bar-wrap { margin: 0.8rem 0; }
  .vote-bar-labels {
    display: flex;
    justify-content: space-between;
    font-family: var(--font-mono);
    font-size: 0.65rem;
    color: var(--text-dim);
    margin-bottom: 0.4rem;
  }
  .vote-bar-track {
    height: 8px;
    background: var(--border);
    display: flex;
    overflow: hidden;
  }
  .vote-bar-bull { background: var(--bullish); transition: width 0.8s ease; }
  .vote-bar-bear { background: var(--bearish); transition: width 0.8s ease; }
  .vote-bar-neut { background: var(--neutral); transition: width 0.8s ease; }

  /* ── Journal ── */
  #journal-list {
    list-style: none;
    max-height: 240px;
    overflow-y: auto;
    scrollbar-width: thin;
    scrollbar-color: var(--border) transparent;
  }
  #journal-list li {
    font-family: var(--font-mono);
    font-size: 0.72rem;
    color: var(--text-dim);
    padding: 0.35rem 0;
    border-bottom: 1px solid var(--border);
    line-height: 1.4;
  }
  #journal-list li:last-child { border-bottom: none; }

  /* ── Price chart mini ── */
  #price-chart { width: 100%; height: 80px; }

  /* ── Macro indicators ── */
  .macro-row {
    display: flex;
    justify-content: space-between;
    align-items: center;
    padding: 0.55rem 0;
    border-bottom: 1px solid var(--border);
    font-size: 0.82rem;
  }
  .macro-row:last-child { border-bottom: none; }
  .macro-name { color: var(--text-dim); font-family: var(--font-mono); font-size: 0.75rem; }
  .macro-val { font-family: var(--font-mono); font-size: 0.82rem; }
  .macro-change.up   { color: var(--bullish); }
  .macro-change.down { color: var(--bearish); }

  /* ── Devil's response block ── */
  .devil-block {
    background: var(--surface2);
    border: 1px solid var(--devil);
    padding: 1.2rem 1.4rem;
    font-size: 0.84rem;
    color: var(--text-dim);
    line-height: 1.6;
    margin-top: 1.5rem;
  }
  .devil-block .devil-label {
    font-family: var(--font-mono);
    font-size: 0.62rem;
    letter-spacing: 0.18em;
    color: var(--devil);
    text-transform: uppercase;
    margin-bottom: 0.6rem;
  }

  /* ── Loading state ── */
  .skeleton {
    background: linear-gradient(90deg, var(--surface2) 25%, var(--border) 50%, var(--surface2) 75%);
    background-size: 200% 100%;
    animation: shimmer 1.5s infinite;
    border-radius: 2px;
    height: 1em;
    margin: 0.4rem 0;
  }
  @keyframes shimmer { 0% {background-position: 200% 0} 100% {background-position: -200% 0} }

  /* ── Timestamp ── */
  #last-updated {
    font-family: var(--font-mono);
    font-size: 0.68rem;
    color: var(--text-dim);
    text-align: right;
    margin-top: 2rem;
  }
</style>
</head>
<body>

<header>
  <div class="logo">⚖ AurumAI <span>/ THE GOLD TRADING COUNCIL</span></div>
  <div id="status-badge">IDLE</div>
</header>

<main>

  <button id="run-btn" onclick="runSession()">▶ CONVENE COUNCIL SESSION</button>

  <div class="grid-3" style="margin-bottom:1.5rem;">
    <button id="paper-buy" onclick="paperTrade('BUY')">BUY XAU/USD</button>
    <button id="paper-sell" onclick="paperTrade('SELL')">SELL XAU/USD</button>
    <button id="paper-close" onclick="paperTrade('CLOSE')">CLOSE POSITION</button>
  </div>

  <div class="card" style="margin-bottom:1.5rem;">
    <div class="card-title">Paper Trading Position</div>
    <div id="paper-position" style="font-family:var(--font-mono);font-size:0.82rem;color:var(--text-dim);line-height:1.6;">No paper position open.</div>
  </div>

  <!-- Verdict Hero -->
  <div id="verdict-hero" class="card">
    <div class="hero-action">
      <div class="action-label">Council Action</div>
      <div class="action-value NEUTRAL" id="hero-action">—</div>
    </div>
    <div class="hero-stat">
      <div class="stat-label">Position Size</div>
      <div class="stat-value gold" id="hero-position">—</div>
    </div>
    <div class="hero-stat">
      <div class="stat-label">Confidence</div>
      <div class="stat-value" id="hero-confidence">—</div>
    </div>
    <div class="rationale" id="hero-rationale">
      Run a council session to see the recommendation.
    </div>
  </div>

  <!-- Row 1: Gold price card + vote tally -->
  <div class="grid-2">

    <div class="card">
      <div class="card-title">Gold Market: XAU/USD Spot</div>
      <div style="display:flex;gap:2rem;margin-bottom:1rem;">
        <div>
          <div class="stat-label">Current Price</div>
          <div class="stat-value gold" id="gold-price">—</div>
        </div>
        <div>
          <div class="stat-label">24h Change</div>
          <div class="stat-value" id="gold-change">—</div>
        </div>
        <div>
          <div class="stat-label">RSI (14)</div>
          <div class="stat-value" id="gold-rsi">—</div>
        </div>
      </div>
      <canvas id="price-chart"></canvas>
    </div>

    <div class="card">
      <div class="card-title">Council Vote Tally</div>
      <div style="display:flex;gap:1.5rem;margin-bottom:1rem;">
        <div><div class="stat-label" style="color:var(--bullish)">Bullish</div><div class="stat-value" id="votes-bull">—</div></div>
        <div><div class="stat-label" style="color:var(--bearish)">Bearish</div><div class="stat-value" id="votes-bear">—</div></div>
        <div><div class="stat-label" style="color:var(--neutral)">Neutral</div><div class="stat-value" id="votes-neut">—</div></div>
      </div>
      <div class="vote-bar-wrap">
        <div class="vote-bar-labels">
          <span>BULLISH</span><span>BEARISH</span><span>NEUTRAL</span>
        </div>
        <div class="vote-bar-track">
          <div class="vote-bar-bull" id="bar-bull" style="width:0%"></div>
          <div class="vote-bar-bear" id="bar-bear" style="width:0%"></div>
          <div class="vote-bar-neut" id="bar-neut" style="width:0%"></div>
        </div>
      </div>
      <div style="margin-top:1rem;">
        <div class="stat-label">Stop Loss Note</div>
        <div style="font-size:0.82rem;color:var(--text-dim);margin-top:0.4rem;line-height:1.5" id="stop-loss">—</div>
      </div>
    </div>

  </div>

  <!-- Row 2: Agent verdicts -->
  <div class="card" style="margin-bottom:1.5rem;">
    <div class="card-title">Specialist Agent Verdicts</div>
    <div class="grid-3" style="margin-bottom:0;" id="agent-cards">
      <!-- populated by JS -->
      <div class="agent-card"><div class="skeleton"></div><div class="skeleton"></div></div>
      <div class="agent-card"><div class="skeleton"></div><div class="skeleton"></div></div>
      <div class="agent-card"><div class="skeleton"></div><div class="skeleton"></div></div>
    </div>
  </div>

  <!-- Row 3: Macro + Journal -->
  <div class="grid-2">

    <div class="card">
      <div class="card-title">Macro Indicators</div>
      <div id="macro-rows"><div class="skeleton"></div><div class="skeleton"></div></div>
    </div>

    <div class="card">
      <div class="card-title">Session Trading Journal</div>
      <ul id="journal-list"><li>No session yet.</li></ul>
    </div>

  </div>

  <p id="last-updated">Last updated: —</p>

</main>

<script>
// ── Mini sparkline chart ──────────────────────────────────────────────────────
function drawSparkline(canvasId, prices) {
  const canvas = document.getElementById(canvasId);
  if (!canvas || !prices || prices.length < 2) return;
  const ctx = canvas.getContext('2d');
  const W = canvas.offsetWidth || 400;
  const H = canvas.offsetHeight || 80;
  canvas.width = W; canvas.height = H;

  const min = Math.min(...prices);
  const max = Math.max(...prices);
  const range = max - min || 1;
  const pad = 6;

  ctx.clearRect(0, 0, W, H);

  // gradient fill
  const grad = ctx.createLinearGradient(0, 0, 0, H);
  grad.addColorStop(0, 'rgba(201,168,76,0.25)');
  grad.addColorStop(1, 'rgba(201,168,76,0.0)');

  const pts = prices.map((p, i) => ({
    x: pad + (i / (prices.length - 1)) * (W - pad * 2),
    y: H - pad - ((p - min) / range) * (H - pad * 2)
  }));

  ctx.beginPath();
  ctx.moveTo(pts[0].x, H);
  pts.forEach(pt => ctx.lineTo(pt.x, pt.y));
  ctx.lineTo(pts[pts.length - 1].x, H);
  ctx.closePath();
  ctx.fillStyle = grad;
  ctx.fill();

  ctx.beginPath();
  ctx.moveTo(pts[0].x, pts[0].y);
  pts.forEach(pt => ctx.lineTo(pt.x, pt.y));
  ctx.strokeStyle = '#C9A84C';
  ctx.lineWidth = 1.5;
  ctx.stroke();

  // last price dot
  const last = pts[pts.length - 1];
  ctx.beginPath();
  ctx.arc(last.x, last.y, 3, 0, Math.PI * 2);
  ctx.fillStyle = '#E8CC7A';
  ctx.fill();
}

// ── Render functions ─────────────────────────────────────────────────────────
function renderHero(data) {
  const final = data.final;
  const action = document.getElementById('hero-action');
  action.textContent = final.action;
  action.className = 'action-value ' + final.final_verdict;

  document.getElementById('hero-position').textContent =
    final.position_size_pct > 0 ? final.position_size_pct + '%' : 'NO TRADE';
  document.getElementById('hero-confidence').textContent = final.confidence_level;
  document.getElementById('hero-rationale').textContent = final.entry_rationale;
  document.getElementById('stop-loss').textContent = final.stop_loss_note;
}

function renderGoldData(data) {
  if (!data.gold_technicals) return;
  try {
    const t = JSON.parse(data.gold_technicals);
    document.getElementById('gold-price').textContent = '$' + t.current_price_usd.toLocaleString();
    const chg = t.change_24h_pct;
    const chgEl = document.getElementById('gold-change');
    chgEl.textContent = (chg >= 0 ? '+' : '') + chg + '%';
    chgEl.style.color = chg >= 0 ? 'var(--bullish)' : 'var(--bearish)';
    const rsiEl = document.getElementById('gold-rsi');
    rsiEl.textContent = t.rsi_14 + ' (' + t.rsi_signal + ')';
    rsiEl.style.color = t.rsi_14 > 70 ? 'var(--bearish)' : t.rsi_14 < 30 ? 'var(--bullish)' : 'var(--text)';
    if (t['10d_price_history']) drawSparkline('price-chart', t['10d_price_history']);
  } catch(e) { console.log('gold parse error', e); }
}

function renderVoteTally(final) {
  const tally = final.vote_tally;
  const total = (tally.BULLISH || 0) + (tally.BEARISH || 0) + (tally.NEUTRAL || 0);
  document.getElementById('votes-bull').textContent = tally.BULLISH || 0;
  document.getElementById('votes-bear').textContent = tally.BEARISH || 0;
  document.getElementById('votes-neut').textContent = tally.NEUTRAL || 0;
  if (total > 0) {
    document.getElementById('bar-bull').style.width = ((tally.BULLISH||0)/total*100) + '%';
    document.getElementById('bar-bear').style.width = ((tally.BEARISH||0)/total*100) + '%';
    document.getElementById('bar-neut').style.width = ((tally.NEUTRAL||0)/total*100) + '%';
  }
}

function renderAgentCards(verdicts) {
  const container = document.getElementById('agent-cards');
  container.innerHTML = '';
  verdicts.forEach(v => {
    const isDevil = v.agent.includes('Devil');
    const conf = v.confidence || 5;
    const div = document.createElement('div');
    div.className = 'agent-card' + (isDevil ? ' devil' : '');
    div.innerHTML = `
      <div class="agent-header">
        <div class="agent-name">${v.agent}</div>
        <div class="agent-verdict ${v.verdict}">${v.verdict}</div>
      </div>
      <div class="agent-key-signal">KEY: ${v.key_signal || '—'}</div>
      <div class="agent-summary">${v.summary}</div>
      <div class="confidence-bar-wrap">
        <div class="confidence-bar-bg">
          <div class="confidence-bar-fill" style="width:${conf*10}%"></div>
        </div>
        <div class="confidence-num">${conf}/10</div>
      </div>
    `;
    container.appendChild(div);
  });
}

function renderMacro(data) {
  if (!data.macro_indicators) return;
  try {
    const m = JSON.parse(data.macro_indicators);
    const container = document.getElementById('macro-rows');
    container.innerHTML = '';
    (m.macro_indicators || []).forEach(ind => {
      if (ind.error) return;
      const up = ind.change_24h_pct >= 0;
      const row = document.createElement('div');
      row.className = 'macro-row';
      row.innerHTML = `
        <div class="macro-name">${ind.name || ind.symbol}</div>
        <div class="macro-val">${ind.current}</div>
        <div class="macro-change ${up ? 'up' : 'down'}">${up ? '+' : ''}${ind.change_24h_pct}%</div>
      `;
      container.appendChild(row);
    });
  } catch(e) { console.log('macro parse error', e); }
}

function renderJournal(journal) {
  const list = document.getElementById('journal-list');
  list.innerHTML = '';
  (journal || []).forEach(entry => {
    const li = document.createElement('li');
    li.textContent = entry;
    list.appendChild(li);
  });
  list.scrollTop = list.scrollHeight;
}


async function paperTrade(action) {
  const res = await fetch('/paper_trade/' + action);
  const data = await res.json();
  document.getElementById('paper-position').textContent =
    data.message + ' | Position: ' + JSON.stringify(data.position);
}

// ── Main: fetch from Flask API ────────────────────────────────────────────────
async function runSession() {
  const btn = document.getElementById('run-btn');
  const badge = document.getElementById('status-badge');
  btn.disabled = true;
  btn.textContent = '⏳ COUNCIL IN SESSION...';
  badge.textContent = 'RUNNING';
  badge.className = 'running';

  try {
    const res = await fetch('/run_session');
    const data = await res.json();

    if (data.error) throw new Error(data.error);

    renderHero(data);
    renderGoldData(data.raw_data);
    renderVoteTally(data.final);
    renderAgentCards(data.final.all_verdicts || []);
    renderMacro(data.raw_data);
    renderJournal(data.final.journal);

    document.getElementById('last-updated').textContent =
      'Last updated: ' + new Date().toUTCString();

    badge.textContent = 'LIVE';
    badge.className = 'live';
  } catch(err) {
    badge.textContent = 'ERROR';
    badge.className = '';
    alert('Session error: ' + err.message);
  } finally {
    btn.disabled = false;
    btn.textContent = '▶ CONVENE COUNCIL SESSION';
  }
}

// Auto-load on page open
window.addEventListener('load', () => {
  fetch('/latest').then(r => r.json()).then(data => {
    if (data && data.final) {
      renderHero(data);
      renderGoldData(data.raw_data);
      renderVoteTally(data.final);
      renderAgentCards(data.final.all_verdicts || []);
      renderMacro(data.raw_data);
      renderJournal(data.final.journal);
      document.getElementById('status-badge').textContent = 'LIVE';
      document.getElementById('status-badge').className = 'live';
    }
  }).catch(() => {});
});
</script>
</body>
</html>

## 🚀 Cell 11 — Flask Server + ngrok (serves the dashboard)

In [ ]:
# ── CELL 11: Flask + ngrok ─────────────────────────────────────────────────────
# Serves the dashboard HTML and exposes /run_session and /latest endpoints.
# ngrok gives us a public HTTPS URL so you can demo from any browser.

from flask import Flask, jsonify, send_file
from pyngrok import ngrok, conf
import threading
import json
from datetime import datetime

app = Flask(__name__)
latest_result = {}  # store the last session result in memory


@app.route("/")
def index():
    return send_file("dashboard.html")


@app.route("/run_session")
def run_session_endpoint():
    """Triggers a full council session and returns results as JSON."""
    global latest_result
    try:
        # Run the async session inside the synchronous Flask context
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        final = loop.run_until_complete(run_council_session())
        loop.close()

        # Collect latest raw MCP data for display
        loop2 = asyncio.new_event_loop()
        asyncio.set_event_loop(loop2)
        raw = loop2.run_until_complete(collect_all_data())
        loop2.close()

        latest_result = {"final": final, "raw_data": raw}
        return jsonify(latest_result)
    except Exception as e:
        return jsonify({"error": str(e)}), 500


@app.route("/latest")
def latest_endpoint():
    """Returns the last session result (so page reload doesn't lose data)."""
    return jsonify(latest_result)



paper_position = {
    "instrument": "XAU/USD",
    "side": None,
    "entry_price": None,
    "status": "FLAT",
}


def latest_xauusd_price():
    try:
        raw = latest_result.get("raw_data", {})
        snapshot = json.loads(raw.get("gold_market_snapshot", "{}"))
        xau = snapshot.get("instruments", {}).get("XAUUSD=X", {})
        return xau.get("current_price")
    except Exception:
        return None


@app.route("/paper_trade/<action>")
def paper_trade(action):
    global paper_position

    action = action.upper()
    price = latest_xauusd_price()

    if price is None:
        return jsonify({
            "message": "Run a council session first so XAU/USD price is available.",
            "position": paper_position,
        })

    if action in ["BUY", "SELL"]:
        paper_position = {
            "instrument": "XAU/USD",
            "side": action,
            "entry_price": price,
            "status": "OPEN",
            "timestamp": datetime.utcnow().strftime("%Y-%m-%d %H:%M UTC"),
        }
        memory.remember_long("paper_trade_open", paper_position)
        return jsonify({
            "message": f"Paper {action} opened at {price}",
            "position": paper_position,
        })

    if action == "CLOSE":
        if paper_position["status"] != "OPEN":
            return jsonify({
                "message": "No open paper position to close.",
                "position": paper_position,
            })

        entry = paper_position["entry_price"]
        side = paper_position["side"]
        if side == "BUY":
            pnl_pct = round(((price - entry) / entry) * 100, 3)
        else:
            pnl_pct = round(((entry - price) / entry) * 100, 3)

        closed = {
            **paper_position,
            "exit_price": price,
            "pnl_pct": pnl_pct,
            "status": "CLOSED",
            "closed_at": datetime.utcnow().strftime("%Y-%m-%d %H:%M UTC"),
        }
        memory.remember_long("paper_trade_closed", closed)
        paper_position = {
            "instrument": "XAU/USD",
            "side": None,
            "entry_price": None,
            "status": "FLAT",
        }
        return jsonify({
            "message": f"Paper trade closed. P/L: {pnl_pct}%",
            "position": closed,
        })

    return jsonify({
        "message": "Unknown paper trade action.",
        "position": paper_position,
    })

# ── Start ngrok ────────────────────────────────────────────────────────────────
conf.get_default().auth_token = os.environ["NGROK_AUTH_TOKEN"]
public_url = ngrok.connect(5000)
print("═" * 55)
print("🌐 Dashboard is LIVE at:")
print(f"   {public_url}")
print("═" * 55)
print("Open this URL in any browser to see your dashboard.")
print("Click 'CONVENE COUNCIL SESSION' to run a live analysis.")
print("Keep this cell running during your demo!")

# ── Start Flask in a background thread ────────────────────────────────────────
# We use threading so the Colab cell doesn't block
def run_flask():
    app.run(port=5000, use_reloader=False, debug=False)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()
print("\n✅ Flask server running on port 5000")

## 🧪 Cell 12 — Optional: Quick test without the browser

In [ ]:
# ── CELL 12: Quick test — run a session and print results to notebook ──────────
# Run this to verify everything works before opening the browser dashboard.

print("Running a test council session in the notebook...\n")
test_result = await run_council_session()

print("\n📋 FULL VERDICT BREAKDOWN:")
print("─" * 45)
for v in test_result["all_verdicts"]:
    bar = "█" * v["confidence"] + "░" * (10 - v["confidence"])
    print(f"{v['agent']:<28} {v['verdict']:<10} {bar} {v['confidence']}/10")

print("\n📓 TRADING JOURNAL:")
for entry in test_result["journal"]:
    print(f"  {entry}")

print("\n⚖️  FINAL ARBITRATOR DECISION:")
print(f"  Action:      {test_result['action']}")
print(f"  Position:    {test_result['position_size_pct']}% of portfolio")
print(f"  Confidence:  {test_result['confidence_level']}")
print(f"  Rationale:   {test_result['entry_rationale']}")
print(f"  Devil said:  {test_result['devil_response']}")
print(f"  Stop loss:   {test_result['stop_loss_note']}")
print(f"  Council:     {test_result['council_summary']}")